# Inversi HVSR single-site berbasis PSO dari file GEOPSY `.hv`

Notebook ini melanjutkan notebook **`3__Hvsr_direct_hv_qc (1).ipynb`**: input tetap satu file `.hv`, parser mempertahankan kurva `Average/Min/Max` dan metadata $f_0$, kemudian kurva diinversi menjadi keluarga model 1D $V_S(z)$.

Metodologi utama mengikuti alur Zaenudin et al. (2024): parameter bebas hanya ketebalan lapisan hingga dan $V_S$; $V_P$ serta densitas dihitung dari relasi Brocher (2005); $Q_P$/$Q_S$ tetap; fungsi objektif adalah norm-2; dan pencarian memakai PSO dengan populasi acak, personal-best, serta global-best. Forward model adalah **rasio fungsi transfer body-wave S/P berinsidensi vertikal pada lapisan 1D viskoelastik**, sesuai asumsi inti ModelHVSR Herak (2008)—**bukan** eliptisitas Rayleigh.

Referensi:

- Zaenudin et al. (2024), DOI: `10.1016/j.eqs.2024.04.004`
- Herak (2008), DOI: `10.1016/j.cageo.2007.07.009`
- Brocher (2005), DOI: `10.1785/0120050017`

> **Batas ilmiah:** file `.hv` hanya memuat kurva ringkasan; preprocessing dan penolakan window tidak dapat diulang. Inversi HVSR tunggal bersifat non-unik. Ensemble di bawah adalah sebaran model ber-misfit rendah dari proses pencarian, bukan posterior Bayesian atau interval kepercayaan terkalibrasi. Bounds contoh harus disesuaikan dengan geologi/lubang bor/geofisika setempat sebelum interpretasi final.


## 1. Import dan temukan root proyek


In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.signal import find_peaks


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan; jalankan notebook dari repo Mikrotremor.")


PROJECT_ROOT = find_project_root(Path.cwd())

import sys

SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from mhvsr_vs30.hvsr_inversion import (
    HVSRCurve,
    InversionBounds,
    acceptable_model_ensemble,
    brocher_density_from_vp,
    brocher_vp_from_vs,
    forward_body_wave_hvsr,
    parse_geopsy_hv,
    run_pso_hvsr_inversion,
    split_particle,
    validate_inversion_bounds,
    vs30_from_layers,
)

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
print(f"Project root: {PROJECT_ROOT}")


## 2. Input pengguna dan search space

Bounds berikut adalah **contoh konservatif untuk demonstrasi**, bukan bounds Bandar Lampung dan bukan hasil inferensi otomatis. Empat lapisan hingga + halfspace menghasilkan total lima lapisan, masih di dalam rentang 5–7 lapisan pada paper. Ubah bounds berdasarkan informasi lokasi Anda.


In [ ]:
# Input utama: file GEOPSY .hv dari notebook QC sebelumnya.
HV_FILE = Path(r"D:\Kulter_2026\hasil_HV\T68.hv")
WINDOW_LENGTH_SECONDS = 30.0  # provenance/QC; harus sama dengan pemrosesan GEOPSY
INVERSION_FREQUENCY_RANGE_HZ = (0.80, 12.0)

# Empat lapisan hingga + satu halfspace. Satuan: meter dan m/s.
THICKNESS_BOUNDS_M = np.array([
    [1.0, 10.0],
    [3.0, 25.0],
    [5.0, 50.0],
    [10.0, 100.0],
])
VS_BOUNDS_M_S = np.array([
    [100.0, 350.0],
    [150.0, 600.0],
    [250.0, 900.0],
    [450.0, 1400.0],
    [800.0, 2500.0],  # halfspace
])

# Paper mengoptimasi h dan Vs; Q dipertahankan tetap.
QP_FIXED = 30.0
QS_FIXED = 10.0
REFERENCE_FREQUENCY_HZ = 1.0
REQUIRE_NONDECREASING_VS = True

# PSO konvensional sesuai persamaan pada ringkasan metodologi pengguna.
PSO_PARTICLES = 50
PSO_ITERATIONS = 80
PSO_SEEDS = [2024, 2025, 2026, 2027]
PSO_INERTIA = 0.72
PSO_LOCAL_ACCELERATION = 1.49
PSO_GLOBAL_ACCELERATION = 1.49
PSO_VELOCITY_LIMIT_FRACTION = 0.25

# Ensemble search: bukan posterior probabilistik.
ENSEMBLE_TOP_N = 200
ENSEMBLE_MAX_RELATIVE_MISFIT = 0.25
DEPTH_STEP_M = 0.5

# Screening defaults yang dapat diubah pengguna; bukan ambang validitas universal.
# Gerbang otomatis hanya menandai kandidat dan review geologi tetap wajib.
MAX_FIT_NRMSE_TO_MEAN = 0.20
MIN_SYNTHETIC_INSIDE_ENVELOPE_FRACTION = 0.80

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "hvsr_single" / HV_FILE.stem
SAVE_OUTPUTS = True

bounds = InversionBounds(THICKNESS_BOUNDS_M, VS_BOUNDS_M_S)
validate_inversion_bounds(bounds)
assert HV_FILE.suffix.lower() == ".hv", "Input harus berupa file .hv"
print(f"Input : {HV_FILE}")
print(f"Output: {OUTPUT_DIR}")


## 3. Baca `.hv`, pilih band inversi, dan ulang QC SESAME yang tersedia


In [ ]:
hv_full = parse_geopsy_hv(HV_FILE)
fmin, fmax = INVERSION_FREQUENCY_RANGE_HZ
band = (hv_full.frequency_hz >= fmin) & (hv_full.frequency_hz <= fmax)
if np.count_nonzero(band) < 20:
    raise ValueError("Band inversi harus berisi sedikitnya 20 sampel frekuensi.")

observed = HVSRCurve(
    frequency_hz=hv_full.frequency_hz[band],
    amplitude=hv_full.amplitude[band],
    minimum=None if hv_full.minimum is None else hv_full.minimum[band],
    maximum=None if hv_full.maximum is None else hv_full.maximum[band],
    source_path=hv_full.source_path,
    number_of_windows=hv_full.number_of_windows,
    number_of_windows_for_f0=hv_full.number_of_windows_for_f0,
    f0_from_average_hz=hv_full.f0_from_average_hz,
    f0_from_windows_hz=hv_full.f0_from_windows_hz,
    f0_min_hz=hv_full.f0_min_hz,
    f0_max_hz=hv_full.f0_max_hz,
    f0_amplitude=hv_full.f0_amplitude,
)

qc = {
    "available": False,
    "reliability_passed": None,
    "clarity_passed": None,
    "reliability_count": None,
    "clarity_count": None,
    "note": "QC SESAME tidak dapat dihitung dari metadata/envelope yang tersedia.",
}
if (
    hv_full.minimum is not None
    and hv_full.maximum is not None
    and hv_full.number_of_windows is not None
    and hv_full.f0_from_windows_hz is not None
    and hv_full.f0_min_hz is not None
    and hv_full.f0_max_hz is not None
):
    from hvsrpy import sesame

    log_f0_std = 0.5 * (np.log(hv_full.f0_max_hz) - np.log(hv_full.f0_min_hz))
    log_f0_mean = np.log(hv_full.f0_from_windows_hz)
    fn_std_normal = float(
        np.sqrt((np.exp(log_f0_std**2) - 1.0) * np.exp(2.0 * log_f0_mean + log_f0_std**2))
    )
    reliability = sesame.reliability(
        windowlength=WINDOW_LENGTH_SECONDS,
        passing_window_count=hv_full.number_of_windows,
        frequency=hv_full.frequency_hz,
        mean_curve=hv_full.amplitude,
        std_curve=hv_full.log_std,
        search_range_in_hz=(None, None),
        verbose=0,
    )
    clarity = sesame.clarity(
        frequency=hv_full.frequency_hz,
        mean_curve=hv_full.amplitude,
        std_curve=hv_full.log_std,
        fn_std=fn_std_normal,
        search_range_in_hz=(None, None),
        verbose=0,
    )
    qc = {
        "available": True,
        "reliability_passed": bool(np.all(reliability)),
        "clarity_passed": bool(np.count_nonzero(clarity) >= 5),
        "reliability_count": int(np.count_nonzero(reliability)),
        "clarity_count": int(np.count_nonzero(clarity)),
        "fn_std_normal_hz": fn_std_normal,
        "note": "Min/Max diperlakukan sebagai envelope lognormal seperti notebook sumber.",
    }

print(f"Titik kurva penuh / inversi : {hv_full.frequency_hz.size} / {observed.frequency_hz.size}")
print(f"Rentang inversi             : {observed.frequency_hz[0]:.3f}–{observed.frequency_hz[-1]:.3f} Hz")
print(f"f0 header                   : {hv_full.f0_from_average_hz} Hz")
print(f"QC SESAME                   : {qc}")


## 4. Inspeksi kurva observasi dan bounds


In [ ]:
fig_input, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
if hv_full.minimum is not None and hv_full.maximum is not None:
    ax.fill_between(hv_full.frequency_hz, hv_full.minimum, hv_full.maximum, alpha=0.25, label="Min–Max")
ax.plot(hv_full.frequency_hz, hv_full.amplitude, color="black", lw=1.8, label="Average H/V")
ax.axvspan(observed.frequency_hz[0], observed.frequency_hz[-1], color="#2563eb", alpha=0.10, label="Band inversi")
if hv_full.f0_from_average_hz is not None:
    ax.axvline(hv_full.f0_from_average_hz, color="#dc2626", ls=":", label=f"f0={hv_full.f0_from_average_hz:.3f} Hz")
ax.set(xscale="log", xlabel="Frekuensi (Hz)", ylabel="H/V", title=f"Input {HV_FILE.name}")
ax.legend()

finite_layers = bounds.finite_layer_count
bounds_table = pd.DataFrame({
    "layer": np.arange(1, finite_layers + 2),
    "type": ["finite"] * finite_layers + ["halfspace"],
    "h_min_m": np.r_[THICKNESS_BOUNDS_M[:, 0], np.nan],
    "h_max_m": np.r_[THICKNESS_BOUNDS_M[:, 1], np.nan],
    "vs_min_m_s": VS_BOUNDS_M_S[:, 0],
    "vs_max_m_s": VS_BOUNDS_M_S[:, 1],
})
display(bounds_table)
ax = axes[1]
for _, row in bounds_table.iloc[:-1].iterrows():
    ax.fill_betweenx([row.h_min_m, row.h_max_m], row.vs_min_m_s, row.vs_max_m_s, alpha=0.25)
ax.set(xlabel="Vs bounds (m/s)", ylabel="Ketebalan tiap lapisan (m)", title="Search bounds (periksa sebelum interpretasi)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Jalankan PSO

Posisi awal acak-uniform di dalam bounds dan velocity awal nol. Update memakai inertia + tarikan local/personal-best + global-best dengan bilangan acak pada setiap langkah. Implementasi ini adalah **PSO konvensional yang tertulis pada ringkasan**, bukan klaim replikasi penuh RR-PSO diskret pada paper.


In [ ]:
results = []
for run_index, seed in enumerate(PSO_SEEDS, start=1):
    run_result = run_pso_hvsr_inversion(
        observed,
        bounds,
        n_particles=PSO_PARTICLES,
        n_iterations=PSO_ITERATIONS,
        seed=seed,
        inertia=PSO_INERTIA,
        local_acceleration=PSO_LOCAL_ACCELERATION,
        global_acceleration=PSO_GLOBAL_ACCELERATION,
        velocity_limit_fraction=PSO_VELOCITY_LIMIT_FRACTION,
        qp=QP_FIXED,
        qs=QS_FIXED,
        reference_frequency_hz=REFERENCE_FREQUENCY_HZ,
        require_nondecreasing_vs=REQUIRE_NONDECREASING_VS,
    )
    results.append(run_result)
    print(
        f"Run {run_index}/{len(PSO_SEEDS)} | seed={seed} | "
        f"L2 awal={run_result.initial_best_misfit:.6f} | "
        f"L2 akhir={run_result.best_misfit:.6f}"
    )

best_run_index = int(np.argmin([item.best_misfit for item in results]))
result = results[best_run_index]
print(f"\nRun terbaik         : {best_run_index + 1} (seed={PSO_SEEDS[best_run_index]})")
print(f"Misfit akhir terbaik: {result.best_misfit:.6f}")
print(f"Total evaluasi model: {sum(item.evaluated_models.shape[0] for item in results):,}")


## 6. Simpan semua evaluasi dan pilih ensemble misfit rendah


In [ ]:
parameter_names = (
    [f"h{i}_m" for i in range(1, finite_layers + 1)]
    + [f"vs{i}_m_s" for i in range(1, finite_layers + 2)]
)
history_frames = []
restart_final_frames = []
for run_index, (seed, run_result) in enumerate(zip(PSO_SEEDS, results, strict=True), start=1):
    history = pd.DataFrame(run_result.evaluated_models, columns=parameter_names)
    history.insert(0, "misfit_l2", run_result.evaluated_misfits)
    history.insert(0, "particle", run_result.evaluated_particle_indices)
    history.insert(0, "iteration", run_result.evaluated_iterations)
    history.insert(0, "seed", seed)
    history.insert(0, "run", run_index)
    history["finite"] = np.isfinite(history["misfit_l2"])
    history_frames.append(history)

    final_models = pd.DataFrame(run_result.models, columns=parameter_names)
    final_models.insert(0, "misfit_l2", run_result.model_misfits)
    final_models.insert(0, "particle", np.arange(PSO_PARTICLES))
    final_models.insert(0, "seed", seed)
    final_models.insert(0, "run", run_index)
    restart_final_frames.append(final_models)

all_models = pd.concat(history_frames, ignore_index=True)
restart_final_models = pd.concat(restart_final_frames, ignore_index=True)
restart_final_models = restart_final_models.loc[
    np.isfinite(restart_final_models["misfit_l2"])
].copy()
unique_restart_models = (
    restart_final_models.sort_values("misfit_l2").drop_duplicates(parameter_names)
)
ensemble = acceptable_model_ensemble(
    unique_restart_models[parameter_names].to_numpy(),
    unique_restart_models["misfit_l2"].to_numpy(),
    top_n=ENSEMBLE_TOP_N,
    maximum_relative_misfit=ENSEMBLE_MAX_RELATIVE_MISFIT,
)
best_models = pd.DataFrame(ensemble.models, columns=parameter_names)
best_models.insert(0, "misfit_l2", ensemble.misfits)
best_models.insert(0, "rank", np.arange(1, ensemble.n_models + 1))
print(f"Model final multi-start unik : {len(unique_restart_models):,}")
print(f"Model search diterima        : {ensemble.n_models}")
print("Sebaran berikut adalah multi-start optimizer-search dispersion, bukan uncertainty posterior.")
display(best_models.head(10))


## 7. Model terbaik, profil ensemble, Vs30 matematis, dan validasi $f_0$


In [ ]:
best_h, best_vs = split_particle(result.best_particle, finite_layers)
best_vp = brocher_vp_from_vs(best_vs)
best_density = brocher_density_from_vp(best_vp)
best_synthetic = forward_body_wave_hvsr(
    observed.frequency_hz,
    best_h,
    best_vs,
    qp=QP_FIXED,
    qs=QS_FIXED,
    reference_frequency_hz=REFERENCE_FREQUENCY_HZ,
)

tops = np.r_[0.0, np.cumsum(best_h)]
bottoms = np.r_[np.cumsum(best_h), np.nan]
layer_table = pd.DataFrame({
    "layer": np.arange(1, finite_layers + 2),
    "top_depth_m": tops,
    "bottom_depth_m": bottoms,
    "thickness_m": np.r_[best_h, np.inf],
    "vs_m_s": best_vs,
    "vp_m_s": best_vp,
    "density_kg_m3": best_density,
    "qp": QP_FIXED,
    "qs": QS_FIXED,
})

maximum_profile_depth = max(30.0, float(np.max(np.sum(ensemble.models[:, :finite_layers], axis=1))))
depth_grid = np.arange(0.0, maximum_profile_depth + DEPTH_STEP_M, DEPTH_STEP_M)
depth_profile = ensemble.depth_profile(depth_grid)
depth_table = pd.DataFrame({
    "depth_m": depth_profile.depth_m,
    "vs_mean_m_s": depth_profile.vs_mean_m_s,
    "vs_std_m_s": depth_profile.vs_std_m_s,
    "vs_search_p05_m_s": depth_profile.vs_p05_m_s,
    "vs_search_median_m_s": depth_profile.vs_median_m_s,
    "vs_search_p95_m_s": depth_profile.vs_p95_m_s,
})

ensemble_vs30 = np.array([
    vs30_from_layers(*split_particle(model, finite_layers)) for model in ensemble.models
])
best_vs30 = vs30_from_layers(best_h, best_vs)

observed_f0 = hv_full.f0_from_average_hz
if observed_f0 is None:
    observed_f0 = float(observed.frequency_hz[np.argmax(observed.amplitude)])
peak_prominence = max(0.05, 0.05 * float(np.ptp(best_synthetic)))
peak_indices, peak_properties = find_peaks(best_synthetic, prominence=peak_prominence)
synthetic_peak_frequencies = observed.frequency_hz[peak_indices]
synthetic_peak_amplitudes = best_synthetic[peak_indices]
observed_f0_inside_band = bool(observed.frequency_hz[0] <= observed_f0 <= observed.frequency_hz[-1])
if peak_indices.size and observed_f0_inside_band:
    matched_peak_position = int(
        np.argmin(np.abs(np.log(synthetic_peak_frequencies / observed_f0)))
    )
    synthetic_f0 = float(synthetic_peak_frequencies[matched_peak_position])
    matched_peak_amplitude = float(synthetic_peak_amplitudes[matched_peak_position])
    dominant_peak_position = int(np.argmax(synthetic_peak_amplitudes))
    dominant_synthetic_f0 = float(synthetic_peak_frequencies[dominant_peak_position])
    competing_peak = bool(
        peak_indices.size > 1
        and np.max(np.delete(synthetic_peak_amplitudes, matched_peak_position))
        > 1.10 * matched_peak_amplitude
    )
else:
    synthetic_f0 = None
    dominant_synthetic_f0 = None
    competing_peak = False

if synthetic_f0 is None:
    f0_consistent = False
elif hv_full.f0_min_hz is not None and hv_full.f0_max_hz is not None:
    f0_consistent = bool(hv_full.f0_min_hz <= synthetic_f0 <= hv_full.f0_max_hz)
else:
    f0_consistent = bool(abs(synthetic_f0 - observed_f0) / observed_f0 <= 0.20)
f0_validation_passed = f0_consistent and not competing_peak

fit_rmse = float(np.sqrt(np.mean((observed.amplitude - best_synthetic) ** 2)))
fit_nrmse_to_mean = fit_rmse / float(np.mean(observed.amplitude))
if observed.minimum is not None and observed.maximum is not None:
    envelope_fraction = float(np.mean((best_synthetic >= observed.minimum) & (best_synthetic <= observed.maximum)))
else:
    envelope_fraction = None

bound_tolerance_fraction = 0.01
best_values = np.asarray(result.best_particle)
bound_span = bounds.upper - bounds.lower
near_lower = (best_values - bounds.lower) <= bound_tolerance_fraction * bound_span
near_upper = (bounds.upper - best_values) <= bound_tolerance_fraction * bound_span
bound_hits = [
    {"parameter": name, "side": "lower" if is_lower else "upper", "value": float(value)}
    for name, value, is_lower, is_upper in zip(
        parameter_names, best_values, near_lower, near_upper, strict=True
    )
    if is_lower or is_upper
]

display(layer_table)
print(f"Vs30 model terbaik (matematis) : {best_vs30:.2f} m/s")
print(f"Vs30 multi-start search median [P05,P95]: {np.median(ensemble_vs30):.2f} "
      f"[{np.quantile(ensemble_vs30, 0.05):.2f}, {np.quantile(ensemble_vs30, 0.95):.2f}] m/s")
print(f"f0 observasi / sintetik terpilih: {observed_f0:.4f} / {synthetic_f0} Hz")
print(f"f0 sintetik dominan             : {dominant_synthetic_f0}")
print(f"Puncak kompetitor ambigu        : {competing_peak}")
print(f"Validasi f0 lulus               : {f0_validation_passed}")
print(f"RMSE / NRMSE terhadap mean H/V : {fit_rmse:.4f} / {fit_nrmse_to_mean:.4f}")
print(f"Fraksi sintetik dalam envelope : {envelope_fraction}")
print(f"Parameter dekat batas (1%)     : {bound_hits or 'tidak ada'}")


## 8. Plot QC inversi


In [ ]:
residual = observed.amplitude - best_synthetic
figure, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
if observed.minimum is not None and observed.maximum is not None:
    ax.fill_between(observed.frequency_hz, observed.minimum, observed.maximum, color="#93c5fd", alpha=0.30, label="Min–Max")
ax.plot(observed.frequency_hz, observed.amplitude, color="black", lw=2.0, label="Observed")
ax.plot(observed.frequency_hz, best_synthetic, color="#dc2626", lw=1.8, label="Synthetic best")
ax.axvline(observed_f0, color="black", ls=":", alpha=0.8, label="f0 observed")
if synthetic_f0 is not None:
    ax.axvline(synthetic_f0, color="#dc2626", ls="--", alpha=0.8, label="f0 synthetic matched")
ax.set(xscale="log", xlabel="Frekuensi (Hz)", ylabel="H/V", title=f"Fit; L2={result.best_misfit:.4f}")
ax.legend()

ax = axes[0, 1]
ax.axhline(0.0, color="black", lw=0.8)
ax.plot(observed.frequency_hz, residual, color="#7c3aed")
ax.set(xscale="log", xlabel="Frekuensi (Hz)", ylabel="Observed − synthetic", title="Residual")

ax = axes[1, 0]
ax.fill_betweenx(
    depth_table.depth_m,
    depth_table.vs_search_p05_m_s,
    depth_table.vs_search_p95_m_s,
    alpha=0.25,
    label="P05–P95 search dispersion",
)
ax.plot(
    depth_table.vs_search_median_m_s,
    depth_table.depth_m,
    lw=2.0,
    label="Median multi-start search",
)
best_depth_vs = np.array([best_vs[np.searchsorted(np.cumsum(best_h), z, side="right")] for z in depth_grid])
ax.step(best_depth_vs, depth_grid, where="post", color="#dc2626", label="Best")
ax.set(xlabel="Vs (m/s)", ylabel="Kedalaman (m)", title="Vs(z): search ensemble")
ax.invert_yaxis()
ax.legend()

ax = axes[1, 1]
for seed, run_result in zip(PSO_SEEDS, results, strict=True):
    ax.semilogy(
        np.arange(run_result.best_misfit_history.size),
        run_result.best_misfit_history,
        label=f"seed {seed}",
    )
ax.set(xlabel="Iterasi", ylabel="Global-best L2", title="Konvergensi multi-start PSO")
ax.legend()

figure.suptitle(f"Inversi HVSR single-site — {HV_FILE.stem}", fontsize=14)
plt.tight_layout()
plt.show()


## 9. Ekspor artefak dan status ilmiah


In [ ]:
observed_table = pd.DataFrame({
    "frequency_hz": hv_full.frequency_hz,
    "hvsr_average": hv_full.amplitude,
    "hvsr_minimum": np.nan if hv_full.minimum is None else hv_full.minimum,
    "hvsr_maximum": np.nan if hv_full.maximum is None else hv_full.maximum,
    "log_std_diagnostic": np.nan if hv_full.log_std is None else hv_full.log_std,
    "selected_for_inversion": band,
})
fit_table = pd.DataFrame({
    "frequency_hz": observed.frequency_hz,
    "hvsr_observed": observed.amplitude,
    "hvsr_synthetic_best": best_synthetic,
    "residual": residual,
})
convergence_table = pd.concat(
    [
        pd.DataFrame({
            "seed": seed,
            "iteration": np.arange(run_result.best_misfit_history.size),
            "global_best_misfit_l2": run_result.best_misfit_history,
        })
        for seed, run_result in zip(PSO_SEEDS, results, strict=True)
    ],
    ignore_index=True,
)

qc_pass = bool(qc.get("reliability_passed") and qc.get("clarity_passed"))
fit_quality_passed = bool(
    fit_nrmse_to_mean <= MAX_FIT_NRMSE_TO_MEAN
    and envelope_fraction is not None
    and envelope_fraction >= MIN_SYNTHETIC_INSIDE_ENVELOPE_FRACTION
)
automatic_gates_passed = (
    qc_pass and f0_validation_passed and fit_quality_passed and not bound_hits
)
status = "bounded_hvsr_candidate" if automatic_gates_passed else "flagged_exploratory_candidate"
input_sha256 = hashlib.sha256(HV_FILE.read_bytes()).hexdigest()
summary = {
    "run_state": "complete",
    "status": status,
    "site_id": HV_FILE.stem,
    "input_file": str(HV_FILE.resolve()),
    "input_sha256": input_sha256,
    "method": {
        "forward": "1D vertically incident S/P body-wave transfer-function ratio (Herak-style)",
        "forward_reference_doi": "10.1016/j.cageo.2007.07.009",
        "inversion_reference_doi": "10.1016/j.eqs.2024.04.004",
        "optimizer": "four independent bounded conventional-PSO restarts; adaptation, not exact RR-PSO",
        "objective": "unweighted L2 norm of observed minus synthetic HVSR",
        "vp_density": "Brocher (2005) empirical polynomials",
    },
    "configuration": {
        "frequency_range_hz": [float(fmin), float(fmax)],
        "finite_layer_count": finite_layers,
        "thickness_bounds_m": THICKNESS_BOUNDS_M.tolist(),
        "vs_bounds_m_s": VS_BOUNDS_M_S.tolist(),
        "qp_fixed": QP_FIXED,
        "qs_fixed": QS_FIXED,
        "reference_frequency_hz": REFERENCE_FREQUENCY_HZ,
        "require_nondecreasing_vs": REQUIRE_NONDECREASING_VS,
        "particles": PSO_PARTICLES,
        "iterations": PSO_ITERATIONS,
        "seeds": PSO_SEEDS,
        "fit_nrmse_maximum": MAX_FIT_NRMSE_TO_MEAN,
        "minimum_inside_envelope_fraction": MIN_SYNTHETIC_INSIDE_ENVELOPE_FRACTION,
    },
    "qc_sesame": qc,
    "results": {
        "best_misfit_l2": result.best_misfit,
        "ensemble_model_count": ensemble.n_models,
        "observed_f0_hz": observed_f0,
        "synthetic_f0_hz": synthetic_f0,
        "dominant_synthetic_f0_hz": dominant_synthetic_f0,
        "synthetic_f0_competing_peak": competing_peak,
        "f0_validation_passed": f0_validation_passed,
        "fit_rmse": fit_rmse,
        "fit_nrmse_to_observed_mean": fit_nrmse_to_mean,
        "synthetic_inside_min_max_fraction": envelope_fraction,
        "parameters_within_one_percent_of_bound": bound_hits,
        "best_vs30_m_s_mathematical": best_vs30,
        "multistart_search_vs30_median_m_s_mathematical": float(np.median(ensemble_vs30)),
        "multistart_search_vs30_p05_m_s": float(np.quantile(ensemble_vs30, 0.05)),
        "multistart_search_vs30_p95_m_s": float(np.quantile(ensemble_vs30, 0.95)),
        "fit_quality_passed": fit_quality_passed,
    },
    "scientific_warnings": [
        "HVSR-only inversion is non-unique.",
        "P05/P95 and standard deviation are multi-start optimizer-search dispersion, not uncertainty or confidence intervals.",
        "Min/Max in .hv are used only as a diagnostic envelope, not formal likelihood uncertainty.",
        "Vs30 is mathematical unless the model and data sensitivity independently resolve 30 m depth.",
        "Validate bounds and the 1D vertical-body-wave assumption with local geological/geophysical evidence.",
        "A geological-plausibility decision remains manual and is not inferred from curve fit alone.",
    ],
}

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    observed_table.to_csv(OUTPUT_DIR / "hvsr_observed.csv", index=False)
    all_models.to_csv(OUTPUT_DIR / "all_models.csv", index=False)
    best_models.to_csv(OUTPUT_DIR / "best_models.csv", index=False)
    layer_table.to_csv(OUTPUT_DIR / "best_layer_model.csv", index=False)
    depth_table.to_csv(OUTPUT_DIR / "vs_depth_ensemble.csv", index=False)
    fit_table.to_csv(OUTPUT_DIR / "hvsr_synthetic_fit.csv", index=False)
    convergence_table.to_csv(OUTPUT_DIR / "convergence.csv", index=False)
    (OUTPUT_DIR / "inversion_summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    figure.savefig(OUTPUT_DIR / "inversion_qc.png", dpi=200, bbox_inches="tight")
    print(f"Artefak tersimpan di: {OUTPUT_DIR}")

display(pd.DataFrame([{
    "status": status,
    "best_L2": result.best_misfit,
    "models_in_ensemble": ensemble.n_models,
    "f0_observed_hz": observed_f0,
    "f0_synthetic_hz": synthetic_f0,
    "f0_validation_passed": f0_validation_passed,
    "fit_quality_passed": fit_quality_passed,
    "bound_hits": len(bound_hits),
    "best_vs30_m_s_mathematical": best_vs30,
}]))


## 10. Cara membaca hasil

1. Periksa dulu `inversion_qc.png`: fit global, residual, bentuk $V_S(z)$ ensemble, dan konvergensi.
2. Periksa `best_models.csv` serta lebar P05–P95 di `vs_depth_ensemble.csv`; solusi tunggal yang tampak bagus tidak menghapus non-keunikan.
3. Gunakan hasil hanya untuk eksplorasi bila salah satu screening gate gagal: QC SESAME, validasi puncak $f_0$, NRMSE/fraksi envelope, atau pemeriksaan parameter yang menyentuh bounds. Ambang fit pada sel konfigurasi adalah default screening, bukan batas validitas universal.
4. Jangan menyatakan kedalaman bedrock, Vs30 final, sesar, atau cekungan hanya dari notebook ini. Gunakan constraint independen (bor, MASW, SPAC, refraksi, geologi) sebelum interpretasi final.
